Para abrir o notebook no Google Colab, altere o domínio `github.com` para `githubtocolab.com`

<div class="alert alert-block alert-danger">
Para praticar programação, é importante que você erre, leia as mensagens de erro e tente corrigí-los.
    
Dessa forma, no Google Colab, é importante que você DESATIVE OS RECURSOS DE AUTOCOMPLETAR:

- Menu Ferramentas -> Configurações
- Na janela que é aberta:
  - Seção Editor -> Desativar "Mostrar sugestões de preenchimento de código com base no contexto"
  - Seção Assistência de IA -> Desabilitar itens

Na versão em inglês:

- Menu Tools -> Settings
- Na janela que é aberta:
  - Seção Editor -> Desativar "Show context-powered code completions"
  - Seção AI Assistance -> Desabilitar itens
</div>

# Exercício 1 - Regressão Linear

Neste exercício, você vai construir um modelo usando a técnica de mínimos quadrados para estimar a concentração de monóxido de carbono (CO) a partir de medições de sensores ambientais, utilizando o conjunto de dados [`air_quality.csv`](./air_quality.csv). Esse banco de dados contém leituras horárias coletadas por sensores de qualidade do ar, incluindo variáveis associadas a diferentes poluentes (sensores e medições de gases) e também variáveis meteorológicas como temperatura (T), umidade relativa (RH) e umidade absoluta (AH).

Ao trabalhar com dados reais, muitas vezes é necessário realizar algumas etapas de filtragem e limpeza para usar os dados para construir um modelo. No caso desse banco de dados, há dois pontos importantes a levados tomados em consideração: (i) alguns valores de sensores aparecem como -200, o que representa alguma medição inválida e devem ser tratados como *missing value* e (ii) como o conjunto de dados consiste em uma série temporal, não deve ser utilizado nenhum tipo de embaralhamento, sendo necessário respeitar a ordem temporal.

No exercício, você irá carregar o dataset com o [Pandas](https://pandas.pydata.org/), tratar valores ausentes,construir um índice temporal juntando as colunas de data e hora, criando um objeto `DateTime` e criar variáveis de tempo (hora, dia e mês).

Após a instalação da biblioteca Pandas e o download do arquivo CSV, é possível carregar os dados com os seguintes comandos:

Obs.: *Caso esteja usando o Google Colab, é necessário fazer o upload do arquivo para o Colab, para que o `pd.read_csv` consiga carregar os dados.*

In [1]:
import pandas as pd
air_data = pd.read_csv(
    "air_quality.csv",
    sep=",",
    decimal=".",
    na_values=["", " ", "NA", -200]
)
print("Colunas originais:", air_data.columns.tolist())

Colunas originais: ['Date', 'Time', 'CO', 'Poluente_1', 'Poluente_2', 'Poluente_3', 'Poluente_4', 'Poluente_5', 'Poluente_6', 'Poluente_7', 'Poluente_8', 'Poluente_9', 'T', 'RH', 'AH']


Com esses comandos, você vai criar um *DataFrame* do Pandas chamado `air_data` contendo os dados do arquivo CSV. A partir deste *DataFrame*, você pode gerar os *arrays* NumPy que serão usados para calcular os parâmetros do modelo de regressão linear.

Para ver algumas linhas do banco de dados, você pode usar o método `.head()`:

In [2]:
air_data.head()

,Date,Time,CO,Poluente_1,Poluente_2,Poluente_3,Poluente_4,Poluente_5,Poluente_6,Poluente_7,Poluente_8,Poluente_9,T,RH,AH
0,10/03/2004,18.00.00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578
1,10/03/2004,19.00.00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255
2,10/03/2004,20.00.00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502
3,10/03/2004,21.00.00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867
4,10/03/2004,22.00.00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888


 A descrição dos dados é a seguinte:


| Coluna        | Descrição |
|---------------|------------|
| `Date`        | Data da medição no formato dia/mês/ano. |
| `Time`        | Horário da medição no formato hora:minuto:segundo. |
| `CO`          | Concentração de monóxido de carbono (variável alvo do modelo). |
| `Poluente_n`  | Medição de poluente atmosférico. |
| `T`           | Temperatura ambiente (°C). |
| `RH`          | Umidade relativa do ar (%). |
| `AH`          | Umidade absoluta do ar. |            |


No conjunto de dados, as informações temporais estão originalmente separadas nas colunas `Date` e `Time`. Para que o modelo possa explorar corretamente a estrutura temporal dos dados, é necessário realizar algumas etapas de processamento dessas colunas. Primeiramente, é interessante remover as linhas onde não existam dados e concatenar as colunas para convertê-las em um objeto no formato `datetime`:

In [3]:
air_data = air_data.dropna(subset=["Date", "Time"])
air_data["DateTime"] = pd.to_datetime(
    air_data["Date"] + " " + air_data["Time"],
    format="%d/%m/%Y %H.%M.%S",
    )
air_data = air_data.dropna(subset=["DateTime"])

Em seguida, podemos configurar a nova coluna como sendo o índice do *DataFrame* e remover as colunas originais `Date` e `Time`:

In [4]:
air_data = air_data.set_index("DateTime")
air_data = air_data.drop(columns=["Date", "Time"])

Finalmente, podemos criar novas variáveis derivadas a partir da coluna `DateTime`:

In [5]:
air_data["Hora"] = air_data.index.hour
air_data["Dia"] = air_data.index.day
air_data["Mes"] = air_data.index.month

A proposta do exercício é que você construa um modelo de regressão linear para prever o valor da coluna do poluente `CO` (Monóxido de Carbono) a partir dos dados das demais variáveis. Para tanto, você pode gerar novas variáveis a partir das originais e/ou descartar variáveis caso julgue que não contribuam para o modelo.

Lembre-se que, para obter um modelo com desempenho melhor, você pode criar outras variáveis a partir de transformações e combinações das variáveis originais. Por exemplo, você poderia calcular a interação entre Temperatura e Umidade Relativa realizando uma multiplicação entre as duas colunas (`T`*`RH`), ou pode calcular a razão entre dois poluentes. Além disso, é possível realizar transformações mais complexas nas varíaveis, como a utilização de amostras passadas do poluente objetivo, visando um [modelo autorregressivo (AR)](https://otexts.com/fpp3/AR.html), ou utilizar métodos de suavização como é o caso da [média móvel (*moving average* - MA)](https://otexts.com/fpp3/moving-averages.html).

Essas variáveis devem ser incluídas como novas colunas no *DataFrame*. Seguem exemplos das sugestões citadas anteriormente:

```python
# 1) Uso de amostras passadas — AR
df["CO_lag1"] = df["CO"].shift(1)   # 1 hora atrás

# 2) Estatísticas móveis do CO
df["CO_media_movel_3h"] = df["CO"].rolling(window=3).mean()
df["CO_std_movel_3h"] = df["CO"].rolling(window=3).std()

# 3) Interações
df["Temp_x_UmidadeRelativa"] = df["T"] * df["RH"]

# Razão (Poluente_5 / Poluente_7)
df["Razao_P5_P7"] = df["Poluente_5"] / (df["Poluente_7"])

```

A base de dados deve ser dividida em um conjunto de treinamento (utilizado para treinar o modelo) e um conjunto de teste (utilizado para avaliar a capacidade de generalização do modelo). Utilize 70% dos dados para o treinamento e 30% para o teste. Para realizar essa divisão, podem ser utilizadas as seguintes linhas de código.

```python
tamanho_treino = int(0.7 * len(air_data))
X_treino, X_teste = X.iloc[:tamanho_treino], X.iloc[tamanho_treino:]
y_treino, y_teste = y.iloc[:tamanho_treino], y.iloc[tamanho_treino:]
```


Em resumo, para obter o vetor $\mathbf{w}_o$ com os coeficientes do modelo de regressão linear, você deve seguir os seguintes passos:

1. Selecionar as variáveis originais que você vai utilizar no modelo. Lembre-se que a variável `CO` não pode ser utilizada pois é a variável que você deseja prever com o modelo;

2. Transformar as variáveis originais de sua seleção e / ou incluir combinações, caso julgue necessário;

3. **Considerar apenas os dados de treinamento** e obter a matriz $\mathbf{X}$ e o vetor $\mathbf{d}$, que podem ser representados como *arrays* do NumPy. Para obter um *array* do NumPy a partir de um *DataFrame* do Pandas, você pode usar o método ``.to_numpy()``;

4. Usando a matriz $\mathbf{X}$ e o vetor $\mathbf{d}$, calcular o vetor $\mathbf{w}_o$ conforme mostrado na aula.

Após obter os coeficientes $\mathbf{w}_o$ do modelo de regressão linear, você vai utilizá-los para prever o valor estimado de CO usando o conjunto de dados de teste. Para cada linha do banco de dados de teste, calcule o valor predito pelo seu modelo e o erro em relação ao valor da coluna `CO`. Calcule também o erro quadrático médio considerando todo o banco de dados de teste.

Ao final do exercício, você deverá apresentar:

1. Uma descrição das variáveis de entrada que você utilizou como entrada e as justificativas para descartar variáveis ou utilizar transformações e combinações;

2. Os códigos utilizados para calcular o vetor $\mathbf{w}_o$ e o erro quadrático médio de seu modelo, considerando os dados de treinamento;

3. O valor obtido para o erro quadrático médio de seu modelo, considerando os dados de teste;

A sugestão é que seja apresentado um Jupyter Notebook usando a linguagem Python, já que essas são as ferramentas que estamos utilizando nesta parte do curso. No entanto, isso não é obrigatório e você pode usar outra linguagem de programação, caso queira.

## Instruções para entrega

- O exercício pode ser feito em dupla ou individualmente;

- A entrega deve incluir:
  - Um vídeo de no **máximo 40s**, mostrando a resolução do exercício;
  - Os **códigos-fontes** dos programas, preferencialmente organizados em um Jupyter Notebook, descrevendo o experimento e mostrando como foram obtidos os resultados solicitados.

- **A correção será feita baseada no vídeo**. Quando o professor/pesquisador ficar com alguma dúvida, serão consultados os códigos-fonte;

- Sobre o vídeo:
  - **Deve incluir áudio** descrevendo o experimento;
  - Gravem a tela do computador usando celular ou usando algum programa de captura de tela (por exemplo Zoom, Google Meet, ou OBS Studio);
  - No início, **deve aparecer o rosto e algum documento do aluno que gravou o vídeo** (como a carteira USP, RG, CNH, etc);
  - No caso de entrega em dupla, **não é necessário que os dois componentes apareçam no vídeo**. No entanto, alternem o apresentador ao longo das entregas dos exercícios e **não esqueçam de incluir os dois nomes no início do vídeo**.
  - Procurem convencer o espectador do vídeo, que vai corrigir o exercício que fizeram os exercícios computacionais solicitados e que eles estão funcionando corretamente. Tentem fazer um bom aproveitamento do tempo para apresentar os resultados solicitados, **respeitando o limite de 40s e não acelerem a velocidade do vídeo**;

- Sobre os códigos-fonte:
  - **Incluir o nome do(s) aluno(s)** no início do programa;

- Sobre o envio no Moodle:
  - Apenas um aluno de cada dupla deve enviar o vídeo no Moodle;
  - Podem ser enviados o arquivo de vídeo (.mkv, .mp4, .avi, etc.) ou um link para o vídeo (Youtube, Google Drive, etc);
    - No segundo caso, certifiquem-se que todos os professores/pesquisadores (magno.silva@usp.br, hae.kim@usp.br, sergiocaceres01@usp.br, renatocan@lps.usp.br) tenham acesso ao seu vídeo.
  - Não se esqueçam de escrever o nome dos componentes da dupla (ou do único aluno, escrevendo: "exercício feito individualmente") em três lugares diferentes: **no campo "comentários sobre o envio" no Moodle**, **no início do vídeo** e no **início dos códigos-fonte**.



In [15]:
import pandas as pd
import numpy as np
#Feature Engineering
# Justificación: Usamos muestras pasadas (AR) porque la contaminación tiene alta inercia.
air_data["CO_lag1"] = air_data["CO"].shift(1)
# Usamos la interacción T x RH porque el clima afecta la dispersión de los gases
air_data["Temp_x_UmidadeRelativa"] = air_data["T"] * air_data["RH"]

# Descartamos las filas con NaNs
air_data = air_data.dropna()
air_data.head()

,CO,Poluente_1,Poluente_2,Poluente_3,Poluente_4,Poluente_5,Poluente_6,Poluente_7,Poluente_8,Poluente_9,T,RH,AH,Hora,Dia,Mes,CO_lag1,Temp_x_UmidadeRelativa
DateTime,,,,,,,,,,,,,,,,,,
2004-03-10 21:00:00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867,21,10,3,2.2,660.00
2004-03-10 22:00:00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888,22,10,3,2.2,667.52
2004-03-10 23:00:00,1.2,1197.0,38.0,4.7,750.0,89.0,1337.0,96.0,1393.0,949.0,11.2,59.2,0.7848,23,10,3,1.6,663.04
2004-03-11 00:00:00,1.2,1185.0,31.0,3.6,690.0,62.0,1462.0,77.0,1333.0,733.0,11.3,56.8,0.7603,0,11,3,1.2,641.84
2004-03-11 01:00:00,1.0,1136.0,31.0,3.3,672.0,62.0,1453.0,76.0,1333.0,730.0,10.7,60.0,0.7702,1,11,3,1.2,642.00


In [21]:
#Definición de la matriz X (entradas) y el vector d (objetivo)
#Descartamos la variable objetivo 'CO' de las entradas.
colunas_para_descartar = ["CO"]
X_df = air_data.drop(columns=colunas_para_descartar)

X_df.insert(0, 'Bias', 1)

d_df = air_data["CO"]

#División en Entrenamiento (70%) y Prueba (30%) respetando el orden temporal
tamanho_treino = int(0.7 * len(air_data))

X_treino = X_df.iloc[:tamanho_treino]
X_teste = X_df.iloc[tamanho_treino:]

d_treino = d_df.iloc[:tamanho_treino]
d_teste = d_df.iloc[tamanho_treino:]

#Convertimos a arreglos de NumPy
X = X_treino.to_numpy()
d = d_treino.to_numpy()
X_test = X_teste.to_numpy()
d_test = d_teste.to_numpy()

#Cálculo del vector de coeficientes w_o usando algebra Lineal
#Ecuación Normal: (X^T * X) * w_o = X^T * d
R = np.dot(X.T, X)
p = np.dot(X.T, d)

#Resolvemos el sistema lineal (es más estable numéricamente que invertir la matriz)
w_o = np.linalg.solve(R, p)
print(w_o)

[ 1.41956257e+00  1.68739861e-03  4.34679730e-04  1.78772874e-01
 -2.27800082e-03  2.91827754e-03 -4.89696912e-04  2.63871694e-03
 -1.97809219e-04 -5.71021089e-04 -7.66121825e-03  4.47588772e-03
 -8.02017068e-01  3.44465340e-03 -2.25500326e-03  1.95627375e-03
  9.20918651e-02  1.12961076e-04]


In [23]:
#calculo de predicciones y Error Cuadrático Medio (MSE)

#Predicción en conjunto de entrenamiento
pred_treino = np.dot(X, w_o)
mse_treino = np.mean((d - pred_treino)**2)

# Predicción en conjunto de prueba
pred_teste = np.dot(X_test, w_o)
mse_teste = np.mean((d_test - pred_teste)**2)


print(f"Erro Quadrático Médio (Treino): {mse_treino:.4f}")
print(f"Erro Quadrático Médio (Teste):  {mse_teste:.4f}")


Erro Quadrático Médio (Treino): 0.0310
Erro Quadrático Médio (Teste):  0.0871


### Conclusão dos Resultados
O modelo obteve um MSE de **0.0310** no conjunto de treinamento e **0.0871** no conjunto de teste.
O aumento do erro no conjunto de teste é um comportamento esperado em séries temporais (visto que a distribuição de dados futuros pode apresentar variâncias não vistas no passado). Contudo, a magnitude de ambos os erros permanece baixa, indicando que as transformações escolhidas (como o lag autorregressivo de 1h e a interação T x RH) foram eficientes para capturar a dinâmica da concentração de CO sem causar um overfitting severo.